In [2]:
from langchain.agents import create_agent
from IPython.display import Markdown
from dotenv.ipython import load_dotenv

In [3]:
# charger les variables d'environnement à partir du fichier .env
load_dotenv(override=True)

True

In [4]:
from langchain_groq import ChatGroq


basic_model = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1)

#Moi je n'utilise pas chat
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
advanced_model = llm

Agent Simple & Dynamic Model
--

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

basic_model = ChatGroq(model="llama-3.1-8b-instant")
advanced_model = ChatGroq(model="llama-3.3-70b-versatile")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    env = request.runtime.context.get('env', 'test')
    if env == 'prod':
        model = advanced_model
        print("advanced model selected ....")
    else:
        model = basic_model
        print("basic model selected ....")
        
    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True
)

Agent Memory
---

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    checkpointer=InMemorySaver()
)

DuckDuckGo & Web Search
---

In [8]:
# Correction de la fonction web_search
from duckduckgo_search import DDGS
from langchain.tools import tool

@tool
def web_search(query: str, num_results: int = 5) -> str:
    """Search the web using DuckDuckGo."""
    try:
        print(f"Search tool is called for {query}")
        ddgs = DDGS()
        results = list(ddgs.text(query, max_results=num_results))
        if not results:
            return f"No results found for {query}"
        formatted = [f"Search for {query}:\n"]
        for i, res in enumerate(results, 1):
            formatted.append(f"{i}. **{res.get('title')}**. {res.get('body')}. {res.get('href')}")
        return "\n".join(formatted)  # le prof avait mis 'return' dans la boucle for 
    except Exception as e:
        return f"Error: {e}"

PythonREPL Tool
---

In [12]:
from langchain_experimental.tools import PythonREPLTool

agent_repl = create_agent(
    model=llm,
    tools=[PythonREPLTool()],
    system_prompt="Always use print(...) to output the final results in PythonREPL, otherwise the tool returns an empty string.",
    debug=True
)

C:\Users\essoa\AppData\Local\Temp\ipykernel_37788\2001101211.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool


Cellule RAG : Remplacer OpenAIEmbeddings par HuggingFace (100% gratuit et local)
---

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Télécharge et exécute localement le modèle (rapide et très léger)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

texts = [
    "Je m'appelle Lhoussaine Daoudi. Je suis professeur en informatique.",
    "Je travaille à la FST de Béni Mellal, USMS.",
    "J'ai obtenu mon doctorat d'État en 2025.",
    "En plus de l'informatique, je suis passionné par la musique et la culture.",
    "Je me passionne pour la philosophie.",
    "Je suis originaire de Zagora, une ville du sud-est du Maroc.",
    "J'ai fait mes études au lycée Salah Eddine Al Ayoubi à Tinghir.",
    "Après le baccalauréat, j'ai étudié à la FST d'Errachidia pendant 4 ans.",
]

vector_store = FAISS.from_texts(texts=texts, embedding=embedding_model)
results = vector_store.similarity_search("Nom, prénom et affiliation", k=2)

print(results[0].page_content)
print(results[1].page_content)

C:\Users\essoa\AppData\Local\Temp\ipykernel_37788\2189837830.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\Users\essoa\OneDrive\Desktop\School staff\TP5_Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2743.73it/s]


Je m'appelle Lhoussaine Daoudi. Je suis professeur en informatique.
Je me passionne pour la philosophie.
